# Notebook 0 — Pilot: Cek Struktur HTML Satu Artikel

**Tujuan:** sebelum crawl ribuan artikel, kita periksa **satu** artikel dulu untuk
menemukan: judul ada di class mana, tanggal di mana, isi berita di mana. Dari sini kita
**hardcode** class tersebut di kode crawling.

## Kenapa perlu parsing? (padahal crawling ambil semua)

Crawling mengunduh **HTML mentah** = seluruh halaman (iklan, menu, script, komentar, isi).
Satu artikel bisa ~250.000 karakter, padahal isi beritanya cuma ~1.500 karakter.

- **Crawling** = beli koran utuh (semua halaman + iklan).
- **Parsing** = menggunting artikel yang kita mau dari koran itu.

Jadi parsing mengubah HTML mentah -> data terstruktur (judul, tanggal, isi). Tanpa itu,
kita cuma punya tumpukan teks campur aduk yang tak bisa dianalisis.

In [ ]:
import requests
from bs4 import BeautifulSoup

# satu artikel contoh untuk diperiksa
url = "https://finance.detik.com/moneter/d-8688622/btn-siapkan-pembiayaan-rp-3-triliun-buat-id-food"
headers = {"User-Agent": "Mozilla/5.0 (research crawler)"}

response = requests.get(url, headers=headers, timeout=30)
print("status:", response.status_code)
print("total HTML mentah:", len(response.text), "karakter (sebagian besar BUKAN isi berita)")

soup = BeautifulSoup(response.text, "html.parser")

## 1. Cari class JUDUL

Kita lihat semua tag `<h1>` dan class-nya. Judul artikel biasanya di `<h1>`.

In [ ]:
for h1 in soup.find_all("h1"):
    print("class:", h1.get("class"), "=> teks:", h1.text.strip()[:60])

# Hasil: judul ada di  <h1 class="detail__title">
judul = soup.find("h1", class_="detail__title").text.strip()
print("\nJUDUL:", judul)

## 2. Cari TANGGAL terbit

Tanggal terbit tidak di dalam teks, tapi di **meta tag** `<meta name="publishdate">`
(lebih akurat & presisi detik).

In [ ]:
meta = soup.find("meta", attrs={"name": "publishdate"})
tanggal = meta["content"]   # format: "2026/10/01 22:44:56"
print("TANGGAL TERBIT:", tanggal)

## 3. Cari class ISI BERITA

Kita cari semua `<div>` yang class-nya mengandung kata 'body', lalu lihat mana yang
berisi teks terpanjang (itulah isi berita).

In [ ]:
for div in soup.find_all("div"):
    cls = div.get("class")
    if cls and any("body" in c for c in cls):
        print("class:", cls, "=> panjang teks:", len(div.text.strip()), "char")

# Hasil: isi berita ada di  <div class="detail__body-text">

In [ ]:
# Ambil isi berita: gabung semua paragraf <p> di dalam body
body = soup.find("div", class_="detail__body-text")
paragraf = [p.text.strip() for p in body.find_all("p") if p.text.strip()]
isi = " ".join(paragraf)

print("JUMLAH PARAGRAF:", len(paragraf))
print("JUMLAH KATA    :", len(isi.split()))
print("\nISI (300 char pertama):")
print(isi[:300])

## 4. Ringkasan 1 artikel (hasil parsing)

Setelah tahu class-nya, kita bisa ambil semua field jadi satu dict rapi.

In [ ]:
satu_artikel = {
    "url": url,
    "judul": judul,
    "tanggal": tanggal,
    "jumlah_kata": len(isi.split()),
    "isi": isi,
}
for k, v in satu_artikel.items():
    print(f"{k:12s}: {str(v)[:70]}")

## Kesimpulan pilot

Dari pemeriksaan 1 artikel, kita menemukan class yang akan di-**hardcode** di crawling:

| Field | Lokasi (hardcode di crawler) |
|-------|------------------------------|
| Judul | `<h1 class="detail__title">` |
| Tanggal | `<meta name="publishdate">` |
| Isi berita | `<div class="detail__body-text">` lalu gabung semua `<p>` |

Class ini **stabil** di artikel DetikFinance 2024-2026 (sudah diuji). Lanjut ke
Notebook 1 (crawl) yang memakai class-class ini untuk ribuan artikel.